In [0]:
# Databricks notebook
# Purpose: Download SUSEP BaseCompleta.zip and save it into the Bronze raw ZIP folder

import os
import requests
import certifi
import urllib3
from datetime import datetime, timezone, timedelta

# ============================================================
# 1. Configuration
# ============================================================

url = "https://www2.susep.gov.br/redarq.asp?arq=BaseCompleta%2ezip"

base_bronze_zip_path = "/Volumes/workspace/susep_bronze/lakehouse/lakehouse-susep/bronze/raw/zip"

file_name = "BaseCompleta.zip"

# UTC-3 timestamp
utc_minus_3 = timezone(timedelta(hours=-3))
execution_timestamp = datetime.now(utc_minus_3).strftime("%Y%m%d_%H%M%S")

# Final target folder
target_folder = f"{base_bronze_zip_path}/{execution_timestamp}"

# Final target file path inside Volume
target_file_path = f"{target_folder}/{file_name}"


print(f"Execution timestamp: {execution_timestamp}")
print(f"Target folder: {target_folder}")
print(f"Target file path: {target_file_path}")

In [0]:
# ============================================================
# 2. Create target folder
# ============================================================

dbutils.fs.mkdirs(target_folder)

print(f"Folder created: {target_folder}")


# ============================================================
# 3. Download ZIP file to local driver
# ============================================================

headers = {
    "User-Agent": "Mozilla/5.0"
}

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

response = requests.get(
        url,
        headers=headers,
        stream=True,
        timeout=300,
        verify=False
    )

response.raise_for_status()

# Write directly to Volume path
with open(target_file_path, "wb") as file:
    for chunk in response.iter_content(chunk_size=1024 * 1024):
        if chunk:
            file.write(chunk)

file_size = os.path.getsize(target_file_path)

print(f"File downloaded: {target_file_path}")
print(f"File size: {file_size:,} bytes")

In [0]:
# ============================================================
# 3. Validate saved file in Bronze folder
# ============================================================

display(dbutils.fs.ls(target_folder))

print("Download process completed successfully.")